# 04. Semi-Supervised Encoding

Objetivo: Hierarchical Blending y Target Encoding suavizado (Bayesian Smoothing) utilizando la distribución de probabilidad prior de los ~209 labels (`ATSEG`) aplicada a las variables categóricas.

In [ ]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def apply_bayesian_smoothing_encoding(df: pd.DataFrame, categorical_col: str, target_col: str, weight: float = 100.0) -> pd.DataFrame:
    """
    Calculates smoothed bayesian encodings. Uses prior distribution from labeled data 
    (approx 209 HCPs) and smooths using the global expected mean across the entire dataset structure.
    Assumes target_col has been numerically encoded.
    """
    logger.info(f"Applying Bayesian Smoothing Target Encoding for {categorical_col}.")
    
    if categorical_col not in df.columns or target_col not in df.columns:
        logger.warning(f"Columns missing. Cannot encode {categorical_col}.")
        return df

    # Filter strictly labeled dataset
    df_labeled = df[df[target_col].notnull()]
    
    if df_labeled.empty:
        logger.warning("No labeled data available to build the prior.")
        return df
        
    # Expected mean over labeled prior
    global_mean = df_labeled[target_col].mean()
    
    # Obtain counts and means by categorical group in labeled subset
    agg_stats = df_labeled.groupby(categorical_col)[target_col].agg(['count', 'mean'])
    
    # Formula: (Count * GroupMean + Weight * GlobalMean) / (Count + Weight)
    smooth_encoding = (agg_stats['count'] * agg_stats['mean'] + weight * global_mean) / (agg_stats['count'] + weight)
    
    encoding_dict = smooth_encoding.to_dict()
    
    df_out = df.copy()
    new_col = f"{categorical_col}_smoothed_enc"
    
    # Map target encoding to the whole dataset
    df_out[new_col] = df_out[categorical_col].map(encoding_dict)
    
    # Fill unknown/unseen categories natively with the global mean
    df_out[new_col] = df_out[new_col].fillna(global_mean)
    
    return df_out
